## read the result batch sampling files

In [8]:
import h5py
import numpy as np
# to see the structure of the hdf5 file and what keys are available
print("Loading the restults_batch_sampling_0.hdf5")
path = "./dataset/results_batch_00_sampling_0.hdf5"
with h5py.File(path, "r") as f:
    def show(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"[DATASET] {name} shape={obj.shape} dtype={obj.dtype}")
        else:
            print(f"[GROUP] {name}")
    f.visititems(show)

# combine multiple batches of data into one big dictionary of numpy arrays
KEYS = [
    "APs_complete","LANs_complete","angMomentum_complete","aratios_complete",
    "eccs_complete","energy_complete","incs_complete","masses_complete",
    "mratios_complete","periods_complete","positions_complete",
    "smaxes_complete","tAnos_complete","times_complete","velocities_complete"
]

def load_one_h5(path, T=None, dtype=np.float32):
    sl = slice(None) if T is None else slice(0, T)
    out = {}
    with h5py.File(path, "r") as f:
        for k in KEYS:
            if k == "times_complete":
                out[k] = f[k][:, sl].astype(dtype)              # (N,T)
            else:
                out[k] = f[k][:, sl, ...].astype(dtype)
    return out

paths = [
    "./dataset/results_batch_00_sampling_0.hdf5",
    "./dataset/results_batch_01_sampling_0.hdf5",
    "./dataset/results_batch_02_sampling_0.hdf5",
    "./dataset/results_batch_03_sampling_0.hdf5",
    "./dataset/results_batch_04_sampling_0.hdf5",
]

# optional: limit to first T steps for faster loading and prototyping
T = 100

dicts = [load_one_h5(p, T=T) for p in paths]

data = {k: np.concatenate([d[k] for d in dicts], axis=0) for k in KEYS}

# print(data["positions_complete"].shape)  # (N_total, T, 3, 3)
print("data load complete")



Loading the restults_batch_sampling_0.hdf5
[DATASET] APs_complete shape=(8000, 1001, 2) dtype=float64
[DATASET] LANs_complete shape=(8000, 1001, 2) dtype=float64
[DATASET] angMomentum_complete shape=(8000, 1001, 3) dtype=float64
[DATASET] aratios_complete shape=(8000, 1001, 1) dtype=float64
[DATASET] eccs_complete shape=(8000, 1001, 2) dtype=float64
[DATASET] energy_complete shape=(8000, 1001, 1) dtype=float64
[DATASET] incs_complete shape=(8000, 1001, 2) dtype=float64
[DATASET] masses_complete shape=(8000, 1001, 3) dtype=float64
[DATASET] mratios_complete shape=(8000, 1001, 2) dtype=float64
[DATASET] periods_complete shape=(8000, 1001, 2) dtype=float64
[DATASET] positions_complete shape=(8000, 1001, 3, 3) dtype=float64
[DATASET] smaxes_complete shape=(8000, 1001, 2) dtype=float64
[DATASET] tAnos_complete shape=(8000, 1001, 2) dtype=float64
[DATASET] times_complete shape=(8000, 1001) dtype=float64
[DATASET] velocities_complete shape=(8000, 1001, 3, 3) dtype=float64
data load complete


In [9]:
# check if we get the same data as before
N = 8000*5  # 5 files, each with 5000 systems
assert data["positions_complete"].shape == (N, T, 3, 3)
assert data["velocities_complete"].shape == (N, T, 3, 3)
assert data["times_complete"].shape == (N, T)


In [10]:
import pandas as pd
import torch

labels = pd.read_csv("dataset/completed_systems_sampling_0.txt", sep=r"\s+", header=None,
                 names=["index","num_batch","time_unbound","ignore_other_than-1","dignosis"])

# datatype
labels = labels.astype({"index":"int64","num_batch":"int64","time_unbound":"float32","ignore_other_than-1":"float32","dignosis":"int64"})

arr = labels.to_numpy()
print(labels.columns)
print(arr.shape, arr.dtype)
print(arr[:10, :5])  
#tensor = torch.from_numpy(arr)  
#print(tensor.shape, tensor.dtype)


Index(['index', 'num_batch', 'time_unbound', 'ignore_other_than-1',
       'dignosis'],
      dtype='str')
(40000, 5) float64
[[360.           0.          10.           1.2236681    0.        ]
 [179.           0.           5.           1.22473073   0.        ]
 [183.           0.           5.           1.24002266   0.        ]
 [361.           0.          20.           1.27451062   0.        ]
 [178.           0.          30.           1.24214554   0.        ]
 [349.           0.          15.           1.26477361   0.        ]
 [180.           0.          20.           1.25246263   0.        ]
 [176.           0.          20.           1.27163601   0.        ]
 [367.           0.          65.           1.30586576   0.        ]
 [182.           0.          30.           1.32230663   0.        ]]


In [11]:
def nan_report(data, T, max_show=5):
    rep = []
    N = next(iter(data.values())).shape[0]
    for k, a in data.items():
        b = a[:, :T, ...] if (a.ndim >= 2 and a.shape[1] >= T) else a

        nan_mask = ~np.isfinite(b)
        bad = nan_mask.any(axis=tuple(range(1, b.ndim)))
        nan_count = int(bad.sum())

        # collect a few NaN coordinates like (sample, t, ...)
        pos = np.argwhere(nan_mask)
        examples = [tuple(map(int, p)) for p in pos[:max_show]]

        rep.append((k, nan_count, N, examples))

    return sorted(rep, key=lambda x: x[1], reverse=True)

for k, bad, N, examples in nan_report(data, T, max_show=5):
    print(k, bad, "/", N)
    if examples:
        print("  NaN examples (sample, time, ...):", examples)


periods_complete 16847 / 40000
  NaN examples (sample, time, ...): [(8, 11, 1), (8, 12, 1), (8, 13, 1), (8, 14, 1), (8, 15, 1)]
tAnos_complete 14509 / 40000
  NaN examples (sample, time, ...): [(0, 0, 0), (0, 0, 1), (1, 0, 1), (3, 0, 1), (6, 0, 1)]
positions_complete 254 / 40000
  NaN examples (sample, time, ...): [(24599, 79, 1, 1), (24603, 79, 1, 1), (24609, 79, 1, 2), (24613, 79, 1, 1), (24615, 79, 1, 1)]
APs_complete 0 / 40000
LANs_complete 0 / 40000
angMomentum_complete 0 / 40000
aratios_complete 0 / 40000
eccs_complete 0 / 40000
energy_complete 0 / 40000
incs_complete 0 / 40000
masses_complete 0 / 40000
mratios_complete 0 / 40000
smaxes_complete 0 / 40000
times_complete 0 / 40000
velocities_complete 0 / 40000


In [12]:
print(data['tAnos_complete'][6])

[[3.1415927         nan]
 [3.1485708  1.8137956 ]
 [2.750173   2.4954262 ]
 [2.3881905  2.8672698 ]
 [2.8748975  3.1278098 ]
 [2.580946   3.3396297 ]
 [1.3754213  3.6463435 ]
 [1.792298   4.1328583 ]
 [3.4226837  5.8489585 ]
 [3.851665   1.9440333 ]
 [3.421827   2.5737529 ]
 [3.1299884  2.9191175 ]
 [2.291755   3.205454  ]
 [1.8054726  3.4793088 ]
 [2.5167458  3.8300738 ]
 [2.197341   4.394933  ]
 [5.9421415  0.5284491 ]
 [4.3826017  2.1453657 ]
 [3.2916744  2.6236765 ]
 [4.209345   2.8775985 ]
 [4.982884   3.1122427 ]
 [3.5032349  3.3225744 ]
 [3.6609762  3.6403673 ]
 [4.4010825  4.043668  ]
 [3.8746974  4.8613825 ]
 [3.3626359  1.456494  ]
 [3.8534665  2.2977934 ]
 [4.0373735  2.6513245 ]
 [3.178712   2.9436831 ]
 [3.2180617  3.2168968 ]
 [3.4133544  3.4728513 ]
 [3.123018   3.8333633 ]
 [2.826098   4.586317  ]
 [2.33709    0.8141357 ]
 [2.7541223  2.1614845 ]
 [2.7360203  2.6340165 ]
 [1.5826184  2.8945193 ]
 [2.1341639  3.1445787 ]
 [2.9680486  3.4082038 ]
 [0.19369389 3.6936517 ]


### A) Phase Variations (“tAnos”)
- Each system is simulated under **4 different starting phases**, including **0° and 180° shifts**.
- Since orbital phases evolve continuously, we **exclude phase variables** from the initial static ML feature vector.

### B) Sensitivity / Chaos Perturbations
- Some systems include a **tiny perturbation in semimajor axis**, creating nearly identical initial conditions.
- This supports a classic **chaos test**: do nearby trajectories diverge over time?


In [13]:
import numpy as np

# 1) Sort `labels` so its row order matches the sample order in `data` (recommended)
labels = labels.sort_values(["num_batch", "index"], ascending=True).reset_index(drop=True)

# 2) Build a boolean mask of samples to KEEP (length = original N, e.g., 40000)
#    A sample is "bad" if it contains any NaN/Inf within the first T time steps.
bad_pos = ~np.isfinite(data["positions_complete"][:, :T, :, :]).all(axis=(1, 2, 3))
bad_vel = ~np.isfinite(data["velocities_complete"][:, :T, :, :]).all(axis=(1, 2, 3))
mask = ~(bad_pos | bad_vel)

# 3) Apply the same mask to both `data` and `labels` (dropped samples disappear from both)
data = {k: v[mask] for k, v in data.items()}
labels = labels.loc[mask].reset_index(drop=True)

TIME_HORIZON = 5000.0
NUM_TIME_BINS = 64
SOFT_LABEL_SIGMA_BINS = 1.5


def build_soft_time_targets(
    times: np.ndarray,
    t_max: float = TIME_HORIZON,
    num_bins: int = NUM_TIME_BINS,
    sigma_bins: float = SOFT_LABEL_SIGMA_BINS,
):
    """
    Convert scalar `time_unbound` into a soft distribution over time bins.

    - For observed unbound times, use a Gaussian-smoothed target over neighboring bins.
    - For right-censored samples (`time_unbound ~= t_max`), put all mass on the last bin.
    """
    times = np.asarray(times, dtype=np.float32)
    clipped = np.clip(times, 0.0, t_max)

    bin_edges = np.linspace(0.0, t_max, num_bins + 1, dtype=np.float32)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    bin_centers[-1] = t_max

    bin_width = float(t_max) / float(num_bins)
    sigma = max(float(sigma_bins) * bin_width, 1e-6)

    dist = np.exp(-0.5 * ((clipped[:, None] - bin_centers[None, :]) / sigma) ** 2).astype(np.float32)

    censored = np.isclose(clipped, t_max, rtol=0, atol=1e-6)
    dist[censored] = 0.0
    dist[censored, -1] = 1.0

    dist /= dist.sum(axis=1, keepdims=True) + 1e-8
    return clipped, dist, bin_edges, bin_centers, censored.astype(np.int32)


# 4) Generate targets AFTER cleaning.
#    `y_hard` is kept only for compatibility with the old binary-classification cells.
#    `y_soft` is the new training target: a distribution over unbound-time bins.
time_target, y_soft, TIME_BIN_EDGES, TIME_BIN_CENTERS, y_hard = build_soft_time_targets(
    labels["time_unbound"].to_numpy()
)
y = y_hard  # legacy alias for the earlier hard-label experiments

# 5) Sanity check: targets length should match the number of remaining samples in `data`
assert len(y_soft) == data["times_complete"].shape[0]
print("After cleaning:", len(y_soft), "censored:", int(y_hard.sum()))
print("y_soft shape:", y_soft.shape)
print("observed unbound:", int((y_hard == 0).sum()), "censored at horizon:", int((y_hard == 1).sum()))



After cleaning: 39746 censored: 14203
y_soft shape: (39746, 64)
observed unbound: 25543 censored at horizon: 14203


In [14]:
import numpy as np

N = len(y_hard)
test_size = 5000
rng = np.random.default_rng(42)

# Keep the split roughly stratified by censored vs observed cases.
pos_idx = np.where(y_hard == 1)[0]
neg_idx = np.where(y_hard == 0)[0]

n_pos = int(round(test_size * len(pos_idx) / N))
n_neg = test_size - n_pos

n_pos = min(n_pos, len(pos_idx))
n_neg = min(n_neg, len(neg_idx))
remain = test_size - (n_pos + n_neg)
if remain > 0:
    if len(pos_idx) - n_pos >= remain:
        n_pos += remain
    else:
        n_neg += remain

test_pos = rng.choice(pos_idx, size=n_pos, replace=False)
test_neg = rng.choice(neg_idx, size=n_neg, replace=False)

test_idx = np.concatenate([test_pos, test_neg])
rng.shuffle(test_idx)

train_mask = np.ones(N, dtype=bool)
train_mask[test_idx] = False

data_train = {k: v[train_mask] for k, v in data.items()}
data_test  = {k: v[test_idx]    for k, v in data.items()}

labels_train = labels.loc[train_mask].reset_index(drop=True)
labels_test  = labels.loc[test_idx].reset_index(drop=True)

# Legacy hard labels for the older classifier cells.
y_train = y_hard[train_mask]
y_test  = y_hard[test_idx]

# New soft targets for time-distribution prediction.
y_soft_train = y_soft[train_mask]
y_soft_test  = y_soft[test_idx]
time_train = time_target[train_mask]
time_test  = time_target[test_idx]

print("train hard-label stats:", len(y_train), "censored:", int(y_train.sum()))
print("test  hard-label stats:", len(y_test),  "censored:", int(y_test.sum()))
print("soft target train/test:", y_soft_train.shape, y_soft_test.shape)
print("time range train:", float(time_train.min()), float(time_train.max()))



train hard-label stats: 34746 censored: 12416
test  hard-label stats: 5000 censored: 1787
soft target train/test: (34746, 64) (5000, 64)
time range train: 5.0 5000.0


## SVM

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

class LinearSVM(nn.Module):
    """
    Linear SVM classifier (binary).
    Uses hinge loss: max(0, 1 - y * f(x)), with y in {-1, +1}.
    """
    def __init__(self, in_dim: int):
        super().__init__()
        self.linear = nn.Linear(in_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # returns raw score f(x) (shape: [B])
        return self.linear(x).squeeze(-1)

def hinge_loss(scores: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """
    scores: f(x), shape [B]
    y: labels in {-1, +1}, shape [B]
    """
    return torch.clamp(1.0 - y * scores, min=0.0).mean()

@torch.no_grad()
def predict(scores: torch.Tensor) -> torch.Tensor:
    # return predicted label in {0,1}
    return (scores >= 0).long()

def train_svm(
    X_train: torch.Tensor,
    y_train: torch.Tensor,
    X_val: torch.Tensor | None = None,
    y_val: torch.Tensor | None = None,
    epochs: int = 100,
    batch_size: int = 1024,
    lr: float = 1e-2,
    weight_decay: float = 1e-4,  # L2 regularization strength (SVM style)
    device: str | torch.device = "cpu",
):
    """
    y_train can be {0,1} or {-1,+1}.
    """
    device = torch.device(device)
    X_train = X_train.to(device)
    y_train = y_train.to(device)

    # Convert labels to {-1, +1}
    if y_train.min() >= 0:
        y_train_svm = y_train * 2 - 1
    else:
        y_train_svm = y_train
    y_train_svm = y_train_svm.float()

    in_dim = X_train.shape[1]
    model = LinearSVM(in_dim).to(device)

    ds = TensorDataset(X_train, y_train_svm)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=False)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)

    def l2_penalty(m: nn.Module) -> torch.Tensor:
        w = m.linear.weight
        return (w * w).sum()

  
    for ep in range(1, epochs + 1):
        model.train()
        total = 0.0
        nb = 0

        for xb, yb in dl:
            scores = model(xb)
            loss = hinge_loss(scores, yb) + 0.5 * weight_decay * l2_penalty(model)

            opt.zero_grad(set_to_none=True)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # NEW

            opt.step()
            total += loss.item()
            nb += 1

        msg = f"epoch {ep:03d} | train_loss {(total/nb):.6f}"

        # optional validation
        if X_val is not None and y_val is not None:
            model.eval()
            Xv = X_val.to(device)
            yv = y_val.to(device)
            yv01 = yv.clone()
            if yv01.min() < 0:  # {-1,+1} -> {0,1}
                yv01 = (yv01 > 0).long()
            scores_v = model(Xv)
            pred_v = predict(scores_v)
            acc = (pred_v == yv01.long()).float().mean().item()
            msg += f" | val_acc {acc:.4f}"

        print(msg)

    return model


@torch.no_grad()
def accuracy(model, X, y, device="cpu"):
    model.eval()
    X = X.to(device)
    y = y.to(device)

    # y can be {0,1} or {-1,+1}; convert to {0,1} for acc
    if y.min() < 0:
        y01 = (y > 0).long()
    else:
        y01 = y.long()

    scores = model(X)                 # raw score f(x)
    pred = (scores >= 0).long()       # decision boundary at 0
    acc = (pred == y01).float().mean().item()
    return acc

In [ ]:
X_train = build_features(data_train, T=T, add_time=True, use_geom=True)
X_test  = build_features(data_test,  T=T, add_time=True, use_geom=True)

print("X_train:", X_train.shape, "finite:", np.isfinite(X_train).all())
print("X_test :", X_test.shape,  "finite:", np.isfinite(X_test).all())

# flatten
X_train_2d = torch.from_numpy(X_train).float().reshape(X_train.shape[0], -1)
X_test_2d  = torch.from_numpy(X_test ).float().reshape(X_test.shape[0], -1)

y_train_t = torch.from_numpy(y_train).long()
y_test_t  = torch.from_numpy(y_test ).long()

eps = 1e-8
mu = X_train_2d.mean(dim=0, keepdim=True)
std = X_train_2d.std(dim=0, keepdim=True).clamp_min(eps)

X_train_2d = (X_train_2d - mu) / std
X_test_2d  = (X_test_2d  - mu) / std

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = train_svm(
    X_train=X_train_2d,
    y_train=y_train_t,
    X_val=None,
    y_val=None,
    device=device
)

train_acc = accuracy(model, X_train_2d, y_train_t, device=device)
test_acc  = accuracy(model, X_test_2d,  y_test_t,  device=device)

print("train_acc:", train_acc)
print("test_acc :", test_acc)


X_train: (34746, 36, 100) finite: True
X_test : (5000, 36, 100) finite: True
Using device: cuda
epoch 001 | train_loss 0.848222
epoch 002 | train_loss 0.778142
epoch 003 | train_loss 0.571249
epoch 004 | train_loss 0.525789
epoch 005 | train_loss 0.529648
epoch 006 | train_loss 0.521556
epoch 007 | train_loss 0.516043
epoch 008 | train_loss 0.534508
epoch 009 | train_loss 0.545958
epoch 010 | train_loss 0.527773
epoch 011 | train_loss 0.541351
epoch 012 | train_loss 0.558626
epoch 013 | train_loss 0.532895
epoch 014 | train_loss 0.545292
epoch 015 | train_loss 0.559332
epoch 016 | train_loss 0.810355
epoch 017 | train_loss 0.774918
epoch 018 | train_loss 0.659857
epoch 019 | train_loss 0.536821
epoch 020 | train_loss 0.510687
epoch 021 | train_loss 0.560390
epoch 022 | train_loss 0.755556
epoch 023 | train_loss 0.885803
epoch 024 | train_loss 0.626826
epoch 025 | train_loss 0.536533
epoch 026 | train_loss 0.505606
epoch 027 | train_loss 0.500227
epoch 028 | train_loss 0.496558
epoch 02

## 1D CNN

In [ ]:
"""
End-to-end: build (B, C, T) features from your dataset + train a 1D-CNN
for soft-label time-distribution prediction using the first T=100 steps.

Features included (per time step):
A) Precomputed orbital/physical series (with angle sin/cos):
  - aratios (1)
  - mratios (2)
  - masses (3)
  - eccs (2)
  - smaxes (2)
  - energy (1)
  - angMomentum (3)
  - APs (2) -> sin/cos (4)
  - LANs (2) -> sin/cos (4)
  - incs (2) -> sin/cos (4)

B) Geometry from positions/velocities (translation/rotation invariant):
  - pairwise distances: r12, r13, r23 (3)
  - pairwise relative speed magnitudes: |v12|, |v13|, |v23| (3)
  - pairwise radial speeds: rdot12, rdot13, rdot23 (3)

C) Optional: normalized time channel (1)

Total channels ~ 38 (A+B+time; periods/tAnos excluded by design).
"""

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ---------------------------
# Feature engineering helpers
# ---------------------------

def sincos(x: np.ndarray) -> np.ndarray:
    """x: (..., k) angles in radians. Return (..., 2k) with [sin, cos]."""
    x = x.astype(np.float32)
    return np.concatenate([np.sin(x), np.cos(x)], axis=-1)

def pairwise_geom(pos: np.ndarray, vel: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    """
    pos: (N, T, 3, 3)  -> bodies, xyz
    vel: (N, T, 3, 3)
    Return geom features per step: (N, T, 9)
      [r12,r13,r23, v12,v13,v23, rdot12,rdot13,rdot23]
    """
    x1, x2, x3 = pos[:, :, 0, :], pos[:, :, 1, :], pos[:, :, 2, :]
    v1, v2, v3 = vel[:, :, 0, :], vel[:, :, 1, :], vel[:, :, 2, :]

    d12 = x1 - x2
    d13 = x1 - x3
    d23 = x2 - x3

    u12 = v1 - v2
    u13 = v1 - v3
    u23 = v2 - v3

    r12 = np.linalg.norm(d12, axis=-1)
    r13 = np.linalg.norm(d13, axis=-1)
    r23 = np.linalg.norm(d23, axis=-1)

    v12 = np.linalg.norm(u12, axis=-1)
    v13 = np.linalg.norm(u13, axis=-1)
    v23 = np.linalg.norm(u23, axis=-1)

    rdot12 = (np.sum(d12 * u12, axis=-1)) / (r12 + eps)
    rdot13 = (np.sum(d13 * u13, axis=-1)) / (r13 + eps)
    rdot23 = (np.sum(d23 * u23, axis=-1)) / (r23 + eps)

    geom = np.stack([r12, r13, r23, v12, v13, v23, rdot12, rdot13, rdot23], axis=-1)
    return geom.astype(np.float32)

def build_features(data: dict, T: int = 200, add_time: bool = True, use_geom: bool = True) -> np.ndarray:
    sl = slice(0, T)
    N = data["aratios_complete"].shape[0]

    # ---- (A) Precomputed series ----
    ar = data["aratios_complete"][:, sl, :].astype(np.float32)
    mr = data["mratios_complete"][:, sl, :].astype(np.float32)
    ms = data["masses_complete"][:, sl, :].astype(np.float32)
    ec = data["eccs_complete"][:, sl, :].astype(np.float32)
    sm = data["smaxes_complete"][:, sl, :].astype(np.float32)
    en = data["energy_complete"][:, sl, :].astype(np.float32)
    am = data["angMomentum_complete"][:, sl, :].astype(np.float32)

    # Angles -> sin/cos  (not use tAnos)
    ap = sincos(data["APs_complete"][:, sl, :])
    lan = sincos(data["LANs_complete"][:, sl, :])
    inc = sincos(data["incs_complete"][:, sl, :])

    feats = [ar, mr, ms, ec, sm, en, am, ap, lan, inc]

    # ---- (B) Geometry ----
    if use_geom:
        pos = data["positions_complete"][:, sl, :, :].astype(np.float32)
        vel = data["velocities_complete"][:, sl, :, :].astype(np.float32)
        geom = pairwise_geom(pos, vel)
        feats.append(geom)

    # ---- (C) Time channel ----
    if add_time:
        t = np.linspace(0.0, 1.0, T, dtype=np.float32)[None, :, None]
        t = np.repeat(t, N, axis=0)
        feats.append(t)

    X = np.concatenate(feats, axis=-1)  # (N,T,C)
    X = X.transpose(0, 2, 1)            # (N,C,T)
    return X.astype(np.float32)

# ---------------------------
# Dataset / Model
# ---------------------------

class SeqDataset(Dataset):
    def __init__(self, X: np.ndarray, y_dist: np.ndarray, time_value: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y_dist = torch.tensor(y_dist.astype(np.float32), dtype=torch.float32)
        self.time_value = torch.tensor(time_value.astype(np.float32), dtype=torch.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, i):
        return self.X[i], self.y_dist[i], self.time_value[i]

class CNN1DSoft(nn.Module):
    """1D CNN for soft-label time-bin distribution prediction."""
    def __init__(self, in_ch: int, num_bins: int):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv1d(in_ch, 128, kernel_size=7, padding=3),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Conv1d(128, 256, kernel_size=5, padding=2),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Conv1d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256, num_bins),
        )

    def forward(self, x):
        return self.head(self.backbone(x))

# ---------------------------
# Training / Evaluation
# ---------------------------

def sharpen_distribution(target_dist: torch.Tensor, gamma: float = 1.0, eps: float = 1e-8) -> torch.Tensor:
    if abs(gamma - 1.0) < 1e-8:
        return target_dist
    t = torch.clamp(target_dist, min=eps) ** gamma
    return t / (t.sum(dim=-1, keepdim=True) + eps)

def soft_cross_entropy(logits: torch.Tensor, target_dist: torch.Tensor) -> torch.Tensor:
    log_prob = torch.log_softmax(logits, dim=-1)
    return -(target_dist * log_prob).sum(dim=-1).mean()

def expected_time_from_logits(logits: torch.Tensor, centers_t: torch.Tensor) -> torch.Tensor:
    prob = torch.softmax(logits, dim=-1)
    return (prob * centers_t.unsqueeze(0)).sum(dim=-1)

@torch.no_grad()
def eval_cnn_soft(model, loader, device, time_bin_centers: np.ndarray, time_horizon: float = TIME_HORIZON):
    model.eval()
    all_prob, all_target, all_time = [], [], []

    for xb, yb_dist, tb in loader:
        xb = xb.to(device)

        logits = model(xb)
        prob = torch.softmax(logits, dim=-1)

        all_prob.append(prob.detach().cpu().numpy())
        all_target.append(yb_dist.numpy())
        all_time.append(tb.numpy())

    all_prob = np.concatenate(all_prob, axis=0)
    all_target = np.concatenate(all_target, axis=0)
    all_time = np.concatenate(all_time, axis=0)

    pred_time = all_prob @ np.asarray(time_bin_centers, dtype=np.float32)
    ce = float(-(all_target * np.log(all_prob + 1e-8)).sum(axis=1).mean())
    mae_all = float(np.mean(np.abs(pred_time - all_time)))

    observed_mask = all_time < (time_horizon - 1e-6)
    censored_mask = ~observed_mask

    mae_observed = (
        float(np.mean(np.abs(pred_time[observed_mask] - all_time[observed_mask])))
        if observed_mask.any()
        else float("nan")
    )
    mae_censored = (
        float(np.mean(np.abs(pred_time[censored_mask] - all_time[censored_mask])))
        if censored_mask.any()
        else float("nan")
    )

    metrics = {
        "soft_ce": ce,
        "mae_all": mae_all,
        "mae_observed": mae_observed,
        "mae_censored": mae_censored,
    }
    return metrics, all_prob, pred_time, all_time

def train_cnn_on_trainset(
    data_train: dict,
    y_dist_train: np.ndarray,
    time_train: np.ndarray,
    time_bin_centers: np.ndarray,
    T: int = 100,
    epochs: int = 20,
    batch_size: int = 256,
    lr: float = 1e-3,
    add_time: bool = True,
    use_geom: bool = True,
    label_sharpen_gamma: float = 1.25,
    lambda_mae_observed: float = 0.30,
    lambda_censored_tail: float = 0.10,
):
    X_tr = build_features(data_train, T=T, add_time=add_time, use_geom=use_geom)  # (N,C,T)
    y_dist_tr = np.asarray(y_dist_train).astype(np.float32)
    time_tr = np.asarray(time_train).astype(np.float32)
    assert X_tr.shape[0] == y_dist_tr.shape[0] == time_tr.shape[0]

    mu = X_tr.mean(axis=(0, 2), keepdims=True)
    sd = X_tr.std(axis=(0, 2), keepdims=True) + 1e-6
    X_tr = (X_tr - mu) / sd
    norm = (mu, sd)

    train_loader = DataLoader(
        SeqDataset(X_tr, y_dist_tr, time_tr),
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = CNN1DSoft(in_ch=X_tr.shape[1], num_bins=y_dist_tr.shape[1]).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs, 1), eta_min=lr * 0.1)

    centers_t = torch.tensor(time_bin_centers, dtype=torch.float32, device=device)
    t_horizon = float(TIME_HORIZON)

    for ep in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_ce = 0.0
        total_mae = 0.0
        total_n = 0

        for xb, yb_dist, tb in train_loader:
            xb = xb.to(device)
            yb_dist = yb_dist.to(device)
            tb = tb.to(device)

            opt.zero_grad(set_to_none=True)

            target = sharpen_distribution(yb_dist, gamma=label_sharpen_gamma)
            logits = model(xb)

            ce = soft_cross_entropy(logits, target)
            pred_time = expected_time_from_logits(logits, centers_t)

            observed_mask = tb < (t_horizon - 1e-6)
            if observed_mask.any():
                mae_obs = torch.abs(pred_time[observed_mask] - tb[observed_mask]).mean() / t_horizon
            else:
                mae_obs = torch.tensor(0.0, device=device)

            censored_mask = ~observed_mask
            if censored_mask.any():
                # Penalize censored samples whose expected time is far below horizon.
                tail_penalty = torch.relu(0.97 * t_horizon - pred_time[censored_mask]).mean() / t_horizon
            else:
                tail_penalty = torch.tensor(0.0, device=device)

            loss = ce + lambda_mae_observed * mae_obs + lambda_censored_tail * tail_penalty
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()

            bs = xb.size(0)
            total_loss += float(loss.item()) * bs
            total_ce += float(ce.item()) * bs
            total_mae += float(torch.abs(pred_time - tb).mean().item()) * bs
            total_n += bs

        scheduler.step()

        print(
            f"epoch {ep:02d} | train_loss {total_loss/total_n:.6f} "
            f"| train_soft_ce {total_ce/total_n:.6f} | train_mae {total_mae/total_n:.2f}"
        )

    return model, norm

@torch.no_grad()
def test_cnn(
    model,
    data_test: dict,
    y_dist_test: np.ndarray,
    time_test: np.ndarray,
    norm,
    time_bin_centers: np.ndarray,
    T: int = 100,
    batch_size: int = 512,
    add_time: bool = True,
    use_geom: bool = True,
):
    X_te = build_features(data_test, T=T, add_time=add_time, use_geom=use_geom)
    y_dist_te = np.asarray(y_dist_test).astype(np.float32)
    time_te = np.asarray(time_test).astype(np.float32)

    mu, sd = norm
    X_te = (X_te - mu) / sd

    device = next(model.parameters()).device
    loader = DataLoader(
        SeqDataset(X_te, y_dist_te, time_te),
        batch_size=batch_size,
        shuffle=False,
    )

    return eval_cnn_soft(model, loader, device, time_bin_centers=time_bin_centers)

In [18]:
model, norm = train_cnn_on_trainset(
    data_train=data_train,
    y_dist_train=y_soft_train,
    time_train=time_train,
    time_bin_centers=TIME_BIN_CENTERS,
    T=T,
    epochs=50,
    batch_size=256,
    lr=1e-3,
    use_geom=True,
    add_time=True,
)

metrics, pred_dist_test, pred_time_test, true_time_test = test_cnn(
    model,
    data_test=data_test,
    y_dist_test=y_soft_test,
    time_test=time_test,
    norm=norm,
    time_bin_centers=TIME_BIN_CENTERS,
    T=T,
    use_geom=True,
    add_time=True,
)

print("TEST metrics:", metrics)
print("pred_dist_test shape:", pred_dist_test.shape)
print("pred_time_test[:10]:", np.round(pred_time_test[:10], 2))
print("true_time_test[:10]:", np.round(true_time_test[:10], 2))

epoch 01 | train_soft_ce 2.105234 | train_mae 1074.352
epoch 02 | train_soft_ce 1.893085 | train_mae 848.331
epoch 03 | train_soft_ce 1.856526 | train_mae 800.854
epoch 04 | train_soft_ce 1.840296 | train_mae 778.087
epoch 05 | train_soft_ce 1.818417 | train_mae 745.810
epoch 06 | train_soft_ce 1.805366 | train_mae 726.366
epoch 07 | train_soft_ce 1.796221 | train_mae 712.490
epoch 08 | train_soft_ce 1.782620 | train_mae 690.205
epoch 09 | train_soft_ce 1.774699 | train_mae 679.168
epoch 10 | train_soft_ce 1.765477 | train_mae 667.351
epoch 11 | train_soft_ce 1.759787 | train_mae 659.962
epoch 12 | train_soft_ce 1.753557 | train_mae 652.090
epoch 13 | train_soft_ce 1.743077 | train_mae 636.110
epoch 14 | train_soft_ce 1.737360 | train_mae 629.408
epoch 15 | train_soft_ce 1.733871 | train_mae 623.640
epoch 16 | train_soft_ce 1.727022 | train_mae 611.480
epoch 17 | train_soft_ce 1.718457 | train_mae 601.690
epoch 18 | train_soft_ce 1.714020 | train_mae 593.028
epoch 19 | train_soft_ce 1.

## TCN

In [8]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score


# =========================================================
# 1) Feature engineering helpers (same as yours)
# =========================================================

def sincos(x: np.ndarray) -> np.ndarray:
    """x: (..., k) angles in radians. Return (..., 2k) with [sin, cos]."""
    x = x.astype(np.float32)
    return np.concatenate([np.sin(x), np.cos(x)], axis=-1)

def pairwise_geom(pos: np.ndarray, vel: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    """
    pos: (N, T, 3, 3)  -> bodies, xyz
    vel: (N, T, 3, 3)
    Return geom features per step: (N, T, 9)
      [r12,r13,r23, v12,v13,v23, rdot12,rdot13,rdot23]
    Invariant under translation/rotation (differences + norms/dots).
    """
    x1, x2, x3 = pos[:, :, 0, :], pos[:, :, 1, :], pos[:, :, 2, :]
    v1, v2, v3 = vel[:, :, 0, :], vel[:, :, 1, :], vel[:, :, 2, :]

    d12 = x1 - x2
    d13 = x1 - x3
    d23 = x2 - x3

    u12 = v1 - v2
    u13 = v1 - v3
    u23 = v2 - v3

    r12 = np.linalg.norm(d12, axis=-1)
    r13 = np.linalg.norm(d13, axis=-1)
    r23 = np.linalg.norm(d23, axis=-1)

    v12 = np.linalg.norm(u12, axis=-1)
    v13 = np.linalg.norm(u13, axis=-1)
    v23 = np.linalg.norm(u23, axis=-1)

    rdot12 = (np.sum(d12 * u12, axis=-1)) / (r12 + eps)
    rdot13 = (np.sum(d13 * u13, axis=-1)) / (r13 + eps)
    rdot23 = (np.sum(d23 * u23, axis=-1)) / (r23 + eps)

    geom = np.stack([r12, r13, r23, v12, v13, v23, rdot12, rdot13, rdot23], axis=-1)
    return geom.astype(np.float32)  # (N,T,9)

def build_features(data: dict, T: int = 100, add_time: bool = True, use_geom: bool = True) -> np.ndarray:
    """
    Build X: (N, C, T) from dict.
    NOTE: following your latest choice: NOT using periods_complete, NOT using tAnos_complete.
    """
    sl = slice(0, T)
    N = data["aratios_complete"].shape[0]

    # ---- (A) Precomputed series ----
    ar = data["aratios_complete"][:, sl, :].astype(np.float32)         # (N,T,1)
    mr = data["mratios_complete"][:, sl, :].astype(np.float32)         # (N,T,2)
    ms = data["masses_complete"][:, sl, :].astype(np.float32)          # (N,T,3)
    ec = data["eccs_complete"][:, sl, :].astype(np.float32)            # (N,T,2)
    sm = data["smaxes_complete"][:, sl, :].astype(np.float32)          # (N,T,2)
    en = data["energy_complete"][:, sl, :].astype(np.float32)          # (N,T,1)
    am = data["angMomentum_complete"][:, sl, :].astype(np.float32)     # (N,T,3)

    # Angles -> sin/cos
    ap  = sincos(data["APs_complete"][:, sl, :])                       # (N,T,4)
    lan = sincos(data["LANs_complete"][:, sl, :])                      # (N,T,4)
    inc = sincos(data["incs_complete"][:, sl, :])                      # (N,T,4)

    feats = [ar, mr, ms, ec, sm, en, am, ap, lan, inc]

    # ---- (B) Geometry ----
    if use_geom:
        pos = data["positions_complete"][:, sl, :, :].astype(np.float32)   # (N,T,3,3)
        vel = data["velocities_complete"][:, sl, :, :].astype(np.float32)  # (N,T,3,3)
        geom = pairwise_geom(pos, vel)                                     # (N,T,9)
        feats.append(geom)

    # ---- (C) Time channel ----
    if add_time:
        t = np.linspace(0.0, 1.0, T, dtype=np.float32)[None, :, None]      # (1,T,1)
        t = np.repeat(t, N, axis=0)                                        # (N,T,1)
        feats.append(t)

    X = np.concatenate(feats, axis=-1)  # (N,T,C)
    X = X.transpose(0, 2, 1)            # (N,C,T)
    return X.astype(np.float32)


# =========================================================
# 2) Dataset
# =========================================================

class SeqDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)  # (N,C,T)
        self.y = torch.tensor(y.astype(np.float32), dtype=torch.float32).view(-1, 1)

    def __len__(self): 
        return self.X.shape[0]

    def __getitem__(self, i):
        return self.X[i], self.y[i]


# =========================================================
# 3) TCN Network
# =========================================================

class Chomp1d(nn.Module):
    """Chomp right side to ensure causality (remove future-leaking padding)."""
    def __init__(self, chomp_size: int):
        super().__init__()
        self.chomp_size = int(chomp_size)

    def forward(self, x):
        if self.chomp_size == 0:
            return x
        return x[:, :, :-self.chomp_size].contiguous()


class TemporalBlock(nn.Module):
    """
    Residual block:
      (Conv -> BN -> ReLU -> Dropout -> Chomp) x 2 + residual
    """
    def __init__(
        self,
        in_ch: int,
        out_ch: int,
        kernel_size: int,
        dilation: int,
        dropout: float = 0.2,
        use_batchnorm: bool = True,
    ):
        super().__init__()
        pad = (kernel_size - 1) * dilation

        layers1 = [
            nn.Conv1d(in_ch, out_ch, kernel_size, padding=pad, dilation=dilation),
        ]
        if use_batchnorm:
            layers1.append(nn.BatchNorm1d(out_ch))
        layers1 += [nn.ReLU(), nn.Dropout(dropout), Chomp1d(pad)]

        layers2 = [
            nn.Conv1d(out_ch, out_ch, kernel_size, padding=pad, dilation=dilation),
        ]
        if use_batchnorm:
            layers2.append(nn.BatchNorm1d(out_ch))
        layers2 += [nn.ReLU(), nn.Dropout(dropout), Chomp1d(pad)]

        self.net = nn.Sequential(*(layers1 + layers2))
        self.downsample = None if in_ch == out_ch else nn.Conv1d(in_ch, out_ch, kernel_size=1)
        self.final_relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.final_relu(out + res)


class TCNBackbone(nn.Module):
    """
    Stack TemporalBlocks with dilations: 1,2,4,8,...
    """
    def __init__(
        self,
        in_ch: int,
        channels: list,
        kernel_size: int = 3,
        dropout: float = 0.2,
        use_batchnorm: bool = True,
    ):
        super().__init__()
        blocks = []
        prev = in_ch
        for i, ch in enumerate(channels):
            blocks.append(
                TemporalBlock(
                    in_ch=prev,
                    out_ch=ch,
                    kernel_size=kernel_size,
                    dilation=2 ** i,
                    dropout=dropout,
                    use_batchnorm=use_batchnorm,
                )
            )
            prev = ch
        self.network = nn.Sequential(*blocks)

    def forward(self, x):
        return self.network(x)  # (B, C_last, T)


class TCNClassifier(nn.Module):
    """
    Classification head on top of TCN backbone.
    pool="avg": global average pool over time
    pool="last": take last hidden state (causal-friendly)
    """
    def __init__(
        self,
        in_ch: int,
        tcn_channels: list = (128, 256, 256, 256),  # 4 blocks -> receptive field bigger, good for T=100
        kernel_size: int = 3,
        dropout: float = 0.2,
        head_dropout: float = 0.3,
        pool: str = "avg",
        use_batchnorm: bool = True,
    ):
        super().__init__()
        self.backbone = TCNBackbone(
            in_ch=in_ch,
            channels=list(tcn_channels),
            kernel_size=kernel_size,
            dropout=dropout,
            use_batchnorm=use_batchnorm,
        )
        self.pool = pool
        out_ch = int(tcn_channels[-1])

        self.head = nn.Sequential(
            nn.Dropout(head_dropout),
            nn.Linear(out_ch, 1),
        )

    def forward(self, x):
        h = self.backbone(x)  # (B, C_last, T)
        if self.pool == "last":
            z = h[:, :, -1]
        else:
            z = h.mean(dim=-1)
        return self.head(z)   # logits


# =========================================================
# 4) Train / Eval utils
# =========================================================

def standardize_train_only(X_tr: np.ndarray, X_te: np.ndarray, eps: float = 1e-6):
    """Standardize each channel using training stats over (N,T)."""
    mu = X_tr.mean(axis=(0, 2), keepdims=True)
    sd = X_tr.std(axis=(0, 2), keepdims=True) + eps
    return (X_tr - mu) / sd, (X_te - mu) / sd, (mu, sd)

@torch.no_grad()
def eval_model(model, loader, device):
    model.eval()
    all_prob, all_y = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        logits = model(xb).detach().cpu()
        prob = torch.sigmoid(logits).numpy().ravel()
        all_prob.append(prob)
        all_y.append(yb.numpy().ravel())

    all_prob = np.concatenate(all_prob)
    all_y = np.concatenate(all_y)

    pred = (all_prob >= 0.5).astype(np.int32)
    auc = roc_auc_score(all_y, all_prob) if len(np.unique(all_y)) > 1 else float("nan")
    f1 = f1_score(all_y, pred) if len(np.unique(all_y)) > 1 else float("nan")
    acc = accuracy_score(all_y, pred)
    return auc, f1, acc


# =========================================================
# 5) Training / Testing (drop-in use)
# =========================================================

def train_tcn_on_trainset(
    data_train: dict,
    y_train: np.ndarray,
    data_val: dict | None = None,
    y_val: np.ndarray | None = None,
    T: int = 100,
    epochs: int = 20,
    batch_size: int = 256,
    lr: float = 1e-3,
    add_time: bool = True,
    use_geom: bool = True,
    # TCN hyperparams
    tcn_channels=(128, 256, 256, 256),
    kernel_size: int = 3,
    dropout: float = 0.2,
    head_dropout: float = 0.3,
    pool: str = "avg",
    weight_decay: float = 1e-4,
    clip_grad: float | None = 1.0,
    seed: int = 42,
):
    """
    Returns:
      model, norm_stats (mu, sd)
    """
    # reproducibility
    torch.manual_seed(seed)
    np.random.seed(seed)

    # Build features
    X_tr = build_features(data_train, T=T, add_time=add_time, use_geom=use_geom)  # (N,C,T)
    y_tr = np.asarray(y_train).astype(np.float32)
    assert X_tr.shape[0] == y_tr.shape[0]

    # Optional validation
    if (data_val is not None) and (y_val is not None):
        X_va = build_features(data_val, T=T, add_time=add_time, use_geom=use_geom)
        y_va = np.asarray(y_val).astype(np.float32)
        X_tr, X_va, norm = standardize_train_only(X_tr, X_va)
        val_loader = DataLoader(SeqDataset(X_va, y_va), batch_size=512, shuffle=False)
    else:
        # train-only normalization
        mu = X_tr.mean(axis=(0, 2), keepdims=True)
        sd = X_tr.std(axis=(0, 2), keepdims=True) + 1e-6
        X_tr = (X_tr - mu) / sd
        norm = (mu, sd)
        val_loader = None

    train_loader = DataLoader(SeqDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True, drop_last=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = TCNClassifier(
        in_ch=X_tr.shape[1],
        tcn_channels=tcn_channels,
        kernel_size=kernel_size,
        dropout=dropout,
        head_dropout=head_dropout,
        pool=pool,
        use_batchnorm=True,
    ).to(device)

    # imbalance-aware loss
    pos = float(y_tr.sum())
    neg = float(len(y_tr) - pos)
    pos_weight = torch.tensor([neg / (pos + 1e-8)], device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_auc = -1.0
    best_state = None

    for ep in range(1, epochs + 1):
        model.train()
        total_loss, total_n = 0.0, 0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)

            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()

            if clip_grad is not None:
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=float(clip_grad))

            opt.step()

            bs = xb.size(0)
            total_loss += float(loss.item()) * bs
            total_n += bs

        msg = f"epoch {ep:02d} | train_loss {total_loss/total_n:.6f}"

        # optional validation each epoch
        if val_loader is not None:
            auc, f1, acc = eval_model(model, val_loader, device)
            msg += f" | val_auc {auc:.4f} | val_f1 {f1:.4f} | val_acc {acc:.4f}"

            if np.isfinite(auc) and auc > best_auc:
                best_auc = auc
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(msg)

    # restore best if had validation
    if best_state is not None:
        model.load_state_dict(best_state)

    return model, norm


@torch.no_grad()
def test_tcn(
    model,
    data_test: dict,
    y_test: np.ndarray,
    norm,
    T: int = 100,
    batch_size: int = 512,
    add_time: bool = True,
    use_geom: bool = True,
):
    X_te = build_features(data_test, T=T, add_time=add_time, use_geom=use_geom)  # (N,C,T)
    y_te = np.asarray(y_test).astype(np.float32)

    mu, sd = norm
    X_te = (X_te - mu) / sd

    device = next(model.parameters()).device
    loader = DataLoader(SeqDataset(X_te, y_te), batch_size=batch_size, shuffle=False)

    auc, f1, acc = eval_model(model, loader, device)
    return auc, f1, acc


# =========================================================
# 6) (Optional) Quick end-to-end usage example
# =========================================================
# model, norm = train_tcn_on_trainset(data_train, y_train, T=100, epochs=20)
# auc, f1, acc = test_tcn(model, data_test, y_test, norm, T=100)
# print("TEST:", auc, f1, acc)

In [9]:
model, norm = train_tcn_on_trainset(
    data_train=data_train,
    y_train=y_train,  
    T=T,
    epochs=20,
    batch_size=32,
    lr=1e-3,
    use_geom=True,
    add_time=True,
)

auc, f1, acc = test_tcn(
    model,
    data_test=data_test,
    y_test=y_test,
    norm=norm,
    T=T,
    use_geom=True,
    add_time=True,
)

print("TEST | AUC:", auc, "F1:", f1, "Acc:", acc)


epoch 01 | train_loss 0.314662
epoch 02 | train_loss 0.243306
epoch 03 | train_loss 0.224653
epoch 04 | train_loss 0.216476
epoch 05 | train_loss 0.207020
epoch 06 | train_loss 0.206834
epoch 07 | train_loss 0.198414
epoch 08 | train_loss 0.198907
epoch 09 | train_loss 0.190891
epoch 10 | train_loss 0.190892
epoch 11 | train_loss 0.185540
epoch 12 | train_loss 0.185531
epoch 13 | train_loss 0.180752
epoch 14 | train_loss 0.178604
epoch 15 | train_loss 0.174354
epoch 16 | train_loss 0.170996
epoch 17 | train_loss 0.168194
epoch 18 | train_loss 0.168286
epoch 19 | train_loss 0.164061
epoch 20 | train_loss 0.158193
TEST | AUC: 0.9795084316774922 F1: 0.9155624837620161 Acc: 0.935


## Attention

In [9]:
import math
import torch
import torch.nn as nn

class PositionalEncoding(nn.Module):
    """Standard sinusoidal positional encoding for (B, T, D)."""
    def __init__(self, d_model: int, max_len: int = 1000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)  # (max_len, D)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)  # (max_len,1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)  # even
        pe[:, 1::2] = torch.cos(position * div_term)  # odd
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, D)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B,T,D)
        T = x.size(1)
        return x + self.pe[:, :T, :]

class AttnClassifier(nn.Module):
    """
    Transformer encoder classifier for inputs (B, C, T).
    - token = one time step
    - output = sequence-level binary logit
    """
    def __init__(
        self,
        in_ch: int,
        d_model: int = 256,
        nhead: int = 8,
        num_layers: int = 3,
        dim_ff: int = 512,
        dropout: float = 0.1,
        pooling: str = "mean",  # "mean" or "cls"
        max_len: int = 1000,
    ):
        super().__init__()
        assert pooling in ("mean", "cls")

        self.pooling = pooling
        self.in_proj = nn.Linear(in_ch, d_model)

        if pooling == "cls":
            self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
        else:
            self.cls = None

        self.pos = PositionalEncoding(d_model, max_len=max_len)
        self.drop = nn.Dropout(dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,   # (B,T,D)
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(0.3),
            nn.Linear(d_model, 1),
        )

        # init
        nn.init.xavier_uniform_(self.in_proj.weight)
        nn.init.zeros_(self.in_proj.bias)
        if self.cls is not None:
            nn.init.normal_(self.cls, mean=0.0, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B,C,T) -> (B,T,C)
        x = x.transpose(1, 2)
        x = self.in_proj(x)  # (B,T,D)

        if self.pooling == "cls":
            B = x.size(0)
            cls_tok = self.cls.expand(B, -1, -1)  # (B,1,D)
            x = torch.cat([cls_tok, x], dim=1)    # (B,T+1,D)

        x = self.pos(x)
        x = self.drop(x)
        x = self.encoder(x)  # (B,T(or T+1),D)

        if self.pooling == "cls":
            seq = x[:, 0, :]          # CLS
        else:
            seq = x.mean(dim=1)       # mean pooling over time

        return self.head(seq)         # (B,1)

def train_attn_on_trainset(
    data_train: dict,
    y_train: np.ndarray,
    T: int = 100,
    epochs: int = 50,
    batch_size: int = 64,
    lr: float = 1e-3,
    add_time: bool = True,
    use_geom: bool = True,
    d_model: int = 128,
    nhead: int = 4,
    num_layers: int = 2,
    dim_ff: int = 512,
    dropout: float = 0.1,
    pooling: str = "cls",
    use_amp: bool = True,
):
    """
    Train on (data_train, y_train) only.
    Returns: model, norm=(mu, sd) computed from train set only.
    """
    # Build features from TRAIN set
    X_tr = build_features(data_train, T=T, add_time=add_time, use_geom=use_geom)  # (N,C,T)
    y_tr = np.asarray(y_train).astype(np.float32)
    assert X_tr.shape[0] == y_tr.shape[0], "X_tr and y_tr must have same N"

    # Standardize using TRAIN only
    mu = X_tr.mean(axis=(0, 2), keepdims=True)
    sd = X_tr.std(axis=(0, 2), keepdims=True) + 1e-6
    X_tr = (X_tr - mu) / sd
    norm = (mu, sd)

    train_loader = DataLoader(
        SeqDataset(X_tr, y_tr),
        batch_size=batch_size,
        shuffle=True,
        drop_last=False
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = AttnClassifier(
        in_ch=X_tr.shape[1],
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_ff=dim_ff,
        dropout=dropout,
        pooling=pooling,
        max_len=T + (1 if pooling == "cls" else 0) + 5,
    ).to(device)

    # pos_weight from TRAIN only
    pos = float(y_tr.sum())
    neg = float(len(y_tr) - pos)
    pos_weight = torch.tensor([neg / (pos + 1e-8)], device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and device.type == "cuda"))

    for ep in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_n = 0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(use_amp and device.type == "cuda")):
                logits = model(xb)
                loss = loss_fn(logits, yb)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            bs = xb.size(0)
            total_loss += float(loss.detach().cpu().item()) * bs
            total_n += bs

        print(f"epoch {ep:02d} | train_loss {total_loss/total_n:.6f}")

    return model, norm


@torch.no_grad()
def test_attn(
    model,
    data_test: dict,
    y_test: np.ndarray,
    norm,
    T: int = 100,
    batch_size: int = 256,
    add_time: bool = True,
    use_geom: bool = True,
):
    """
    Evaluate on external test set (data_test, y_test), using train norm (mu, sd).
    Returns: (auc, f1, acc)
    """
    X_te = build_features(data_test, T=T, add_time=add_time, use_geom=use_geom)  # (N,C,T)
    y_te = np.asarray(y_test).astype(np.float32)
    assert X_te.shape[0] == y_te.shape[0], "X_te and y_te must have same N"

    mu, sd = norm
    X_te = (X_te - mu) / sd

    device = next(model.parameters()).device
    test_loader = DataLoader(
        SeqDataset(X_te, y_te),
        batch_size=batch_size,
        shuffle=False,
        drop_last=False
    )

    auc, f1, acc = eval_model(model, test_loader, device)
    return auc, f1, acc

In [10]:

model, norm = train_attn_on_trainset(
    data_train=data_train,
    y_train=y_train,
    T=T,
    epochs=50,
    batch_size=64,
    lr=1e-3,
    use_geom=True,
    add_time=True,
    pooling="cls",
    use_amp=True,
)


auc, f1, acc = test_attn(
    model,
    data_test=data_test,
    y_test=y_test,
    norm=norm,
    T=T,
    use_geom=True,
    add_time=True,
)

print(f"TEST | AUC {auc:.4f} | F1 {f1:.4f} | Acc {acc:.4f}")



/tmp/ipykernel_23776/1703861942.py:61: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
/tmp/ipykernel_23776/1703861942.py:155: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and device.type == "cuda"))
/tmp/ipykernel_23776/1703861942.py:166: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(use_amp and device.type == "cuda")):


epoch 01 | train_loss 0.284543
epoch 02 | train_loss 0.230395
epoch 03 | train_loss 0.215834
epoch 04 | train_loss 0.208172
epoch 05 | train_loss 0.201265
epoch 06 | train_loss 0.201335
epoch 07 | train_loss 0.195673
epoch 08 | train_loss 0.194424
epoch 09 | train_loss 0.194808
epoch 10 | train_loss 0.190424
epoch 11 | train_loss 0.187101
epoch 12 | train_loss 0.183058
epoch 13 | train_loss 0.184706
epoch 14 | train_loss 0.178987
epoch 15 | train_loss 0.180189
epoch 16 | train_loss 0.176776
epoch 17 | train_loss 0.174200
epoch 18 | train_loss 0.172178
epoch 19 | train_loss 0.170911
epoch 20 | train_loss 0.171218
epoch 21 | train_loss 0.167191
epoch 22 | train_loss 0.165835
epoch 23 | train_loss 0.163893
epoch 24 | train_loss 0.162985
epoch 25 | train_loss 0.158297
epoch 26 | train_loss 0.160349
epoch 27 | train_loss 0.156132
epoch 28 | train_loss 0.155080
epoch 29 | train_loss 0.157639
epoch 30 | train_loss 0.151052
epoch 31 | train_loss 0.152866
epoch 32 | train_loss 0.147576
epoch 33

## GNN

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ===========================
# Helpers
# ===========================

def sincos(x: np.ndarray) -> np.ndarray:
    """x: (..., k) angles in radians -> (..., 2k)"""
    x = x.astype(np.float32)
    return np.concatenate([np.sin(x), np.cos(x)], axis=-1)

def build_global_orbital_features(data: dict, T: int) -> np.ndarray:
    """
    Only use the orbital parameters explicitly specified:

      - aratios (2)
      - mratios (1)
      - eccs (2)
      - incs (2) -> sin/cos (4)
      - APs  (2) -> sin/cos (4)
      - LANs (2) -> sin/cos (4)

    Output:
      global_x: (N, T, Fg)

    Total feature dim:
      2 + 1 + 2 + 4 + 4 + 4 = 17
    """
    sl = slice(0, T)

    ar = data["aratios_complete"][:, sl, :].astype(np.float32)
    mr = data["mratios_complete"][:, sl, :].astype(np.float32)
    ec = data["eccs_complete"][:, sl, :].astype(np.float32)

    inc = sincos(data["incs_complete"][:, sl, :])
    ap  = sincos(data["APs_complete"][:, sl, :])
    lan = sincos(data["LANs_complete"][:, sl, :])

    g = np.concatenate([ar, mr, ec, inc, ap, lan], axis=-1).astype(np.float32)
    return g

def build_node_features(data: dict, T: int, use_masses: bool = False) -> np.ndarray:
    """
    Node features per time step and body.
    Output: (N, T, 3, Fn)

    If use_masses=False:
      node features = [position xyz, velocity xyz] -> Fn = 6

    If use_masses=True:
      node features = [mass, position xyz, velocity xyz] -> Fn = 7
    """
    sl = slice(0, T)

    pos = data["positions_complete"][:, sl, :, :].astype(np.float32)
    vel = data["velocities_complete"][:, sl, :, :].astype(np.float32)

    if use_masses:
        masses = data["masses_complete"][:, sl, :].astype(np.float32)
        masses = masses[..., None]
        node = np.concatenate([masses, pos, vel], axis=-1).astype(np.float32)
    else:
        node = np.concatenate([pos, vel], axis=-1).astype(np.float32)

    return node

def build_edge_features(data: dict, T: int, eps: float = 1e-8):
    """
    Build directed edge features for the 3-body fully connected graph (no self loops).

    Return:
      edge_feat: (N, T, E=6, Fe)
      senders: np.ndarray shape (6,)
      receivers: np.ndarray shape (6,)

    Edge features:
      - relative position dx,dy,dz (3)
      - distance ||d|| (1)
      - relative velocity dvx,dvy,dvz (3)
      - relative speed ||dv|| (1)
      - radial speed (1)
    total Fe = 9
    """
    sl = slice(0, T)
    pos = data["positions_complete"][:, sl, :, :].astype(np.float32)
    vel = data["velocities_complete"][:, sl, :, :].astype(np.float32)

    senders = np.array([0, 0, 1, 1, 2, 2], dtype=np.int64)
    receivers = np.array([1, 2, 0, 2, 0, 1], dtype=np.int64)
    E = len(senders)

    edge_feats = []
    for e in range(E):
        i = senders[e]
        j = receivers[e]

        d = pos[:, :, j, :] - pos[:, :, i, :]
        u = vel[:, :, j, :] - vel[:, :, i, :]

        r = np.linalg.norm(d, axis=-1, keepdims=True)
        v = np.linalg.norm(u, axis=-1, keepdims=True)
        rdot = (np.sum(d * u, axis=-1, keepdims=True)) / (r + eps)

        ef = np.concatenate([d, r, u, v, rdot], axis=-1).astype(np.float32)
        edge_feats.append(ef)

    edge_feat = np.stack(edge_feats, axis=2)
    return edge_feat, senders, receivers

def build_graph_data(data: dict, T: int = 100, use_masses: bool = False):
    node_x = build_node_features(data, T, use_masses=use_masses)
    edge_x, senders, receivers = build_edge_features(data, T)
    global_x = build_global_orbital_features(data, T)
    return node_x, edge_x, global_x, senders, receivers

# ===========================
# Dataset
# ===========================

class GraphSeqDataset(Dataset):
    def __init__(self, node_x, edge_x, global_x, y_dist, time_value):
        self.node_x = torch.tensor(node_x, dtype=torch.float32)
        self.edge_x = torch.tensor(edge_x, dtype=torch.float32)
        self.global_x = torch.tensor(global_x, dtype=torch.float32)
        self.y_dist = torch.tensor(y_dist.astype(np.float32), dtype=torch.float32)
        self.time_value = torch.tensor(time_value.astype(np.float32), dtype=torch.float32)

    def __len__(self):
        return self.node_x.shape[0]

    def __getitem__(self, i):
        return self.node_x[i], self.edge_x[i], self.global_x[i], self.y_dist[i], self.time_value[i]

# ===========================
# Graph model
# ===========================

class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return self.net(x)

class GraphEncoderStep(nn.Module):
    def __init__(self, node_dim, edge_dim, global_dim, hidden_dim=128, msg_dim=128, out_dim=128):
        super().__init__()
        self.node_proj = nn.Linear(node_dim + global_dim, hidden_dim)
        self.edge_mlp = MLP(2 * hidden_dim + edge_dim + global_dim, msg_dim, msg_dim)
        self.node_update = MLP(hidden_dim + msg_dim + global_dim, hidden_dim, hidden_dim)
        self.readout = MLP(hidden_dim + global_dim, hidden_dim, out_dim)

    def forward(self, node_x, edge_x, global_x, senders, receivers):
        B, Nn, _ = node_x.shape
        E = edge_x.shape[1]

        g_node = global_x.unsqueeze(1).expand(B, Nn, global_x.shape[-1])
        h = self.node_proj(torch.cat([node_x, g_node], dim=-1))

        h_send = h[:, senders, :]
        h_recv = h[:, receivers, :]
        g_edge = global_x.unsqueeze(1).expand(B, E, global_x.shape[-1])

        m_in = torch.cat([h_send, h_recv, edge_x, g_edge], dim=-1)
        m = self.edge_mlp(m_in)

        agg = torch.zeros(B, Nn, m.shape[-1], device=node_x.device, dtype=node_x.dtype)
        for e, r in enumerate(receivers):
            agg[:, r, :] += m[:, e, :]

        node_upd_in = torch.cat([h, agg, g_node], dim=-1)
        h = self.node_update(node_upd_in)

        h_pool = h.mean(dim=1)
        g_t = self.readout(torch.cat([h_pool, global_x], dim=-1))
        return g_t

class GraphGRUTimeDistPredictor(nn.Module):
    """
    For each time step:
      G_t -> GraphEncoder -> g_t

    Then:
      (g_1,...,g_T) -> GRU -> z -> MLP -> logits over time bins
    """
    def __init__(
        self,
        node_dim,
        edge_dim,
        global_dim,
        num_bins,
        graph_hidden=128,
        msg_dim=128,
        graph_out_dim=128,
        gru_hidden=128,
        num_gru_layers=1,
        dropout=0.3,
    ):
        super().__init__()
        self.graph_encoder = GraphEncoderStep(
            node_dim=node_dim,
            edge_dim=edge_dim,
            global_dim=global_dim,
            hidden_dim=graph_hidden,
            msg_dim=msg_dim,
            out_dim=graph_out_dim,
        )

        self.temporal = nn.GRU(
            input_size=graph_out_dim,
            hidden_size=gru_hidden,
            num_layers=num_gru_layers,
            batch_first=True,
        )

        self.head = nn.Sequential(
            nn.Linear(gru_hidden, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_bins),
        )

    def forward(self, node_x, edge_x, global_x, senders, receivers):
        B, T, _, _ = node_x.shape

        g_seq = []
        for t in range(T):
            g_t = self.graph_encoder(
                node_x[:, t],
                edge_x[:, t],
                global_x[:, t],
                senders,
                receivers,
            )
            g_seq.append(g_t)

        g_seq = torch.stack(g_seq, dim=1)
        _, h_n = self.temporal(g_seq)
        z = h_n[-1]
        return self.head(z)

# ===========================
# Training / Eval
# ===========================

def soft_cross_entropy(logits: torch.Tensor, target_dist: torch.Tensor) -> torch.Tensor:
    log_prob = torch.log_softmax(logits, dim=-1)
    return -(target_dist * log_prob).sum(dim=-1).mean()

@torch.no_grad()
def eval_graph_model(model, loader, device, senders, receivers, time_bin_centers, time_horizon=TIME_HORIZON):
    model.eval()
    all_prob, all_target, all_time = [], [], []

    senders_t = torch.tensor(senders, dtype=torch.long, device=device)
    receivers_t = torch.tensor(receivers, dtype=torch.long, device=device)
    time_bin_centers_t = torch.tensor(time_bin_centers, dtype=torch.float32, device=device)

    for node_x, edge_x, global_x, y_dist, time_value in loader:
        node_x = node_x.to(device)
        edge_x = edge_x.to(device)
        global_x = global_x.to(device)

        logits = model(node_x, edge_x, global_x, senders_t, receivers_t)
        prob = torch.softmax(logits, dim=-1)
        pred_time = (prob * time_bin_centers_t.unsqueeze(0)).sum(dim=-1)

        all_prob.append(prob.detach().cpu().numpy())
        all_target.append(y_dist.numpy())
        all_time.append(time_value.numpy())

    all_prob = np.concatenate(all_prob, axis=0)
    all_target = np.concatenate(all_target, axis=0)
    all_time = np.concatenate(all_time, axis=0)
    pred_time = all_prob @ np.asarray(time_bin_centers, dtype=np.float32)

    ce = float(-(all_target * np.log(all_prob + 1e-8)).sum(axis=1).mean())
    mae_all = float(np.mean(np.abs(pred_time - all_time)))

    observed_mask = all_time < (time_horizon - 1e-6)
    mae_observed = float(np.mean(np.abs(pred_time[observed_mask] - all_time[observed_mask]))) if observed_mask.any() else float("nan")

    metrics = {
        "soft_ce": ce,
        "mae_all": mae_all,
        "mae_observed": mae_observed,
    }
    return metrics, all_prob, pred_time, all_time

def train_graph_model_on_trainset(
    data_train: dict,
    y_dist_train: np.ndarray,
    time_train: np.ndarray,
    time_bin_centers: np.ndarray,
    T: int = 100,
    epochs: int = 20,
    batch_size: int = 256,
    lr: float = 1e-3,
    use_masses: bool = False,
):
    node_tr, edge_tr, global_tr, senders, receivers = build_graph_data(
        data_train, T=T, use_masses=use_masses
    )
    y_dist_tr = np.asarray(y_dist_train).astype(np.float32)
    time_tr = np.asarray(time_train).astype(np.float32)

    node_mu = node_tr.mean(axis=(0, 1, 2), keepdims=True)
    node_sd = node_tr.std(axis=(0, 1, 2), keepdims=True) + 1e-6
    edge_mu = edge_tr.mean(axis=(0, 1, 2), keepdims=True)
    edge_sd = edge_tr.std(axis=(0, 1, 2), keepdims=True) + 1e-6
    global_mu = global_tr.mean(axis=(0, 1), keepdims=True)
    global_sd = global_tr.std(axis=(0, 1), keepdims=True) + 1e-6

    node_tr = (node_tr - node_mu) / node_sd
    edge_tr = (edge_tr - edge_mu) / edge_sd
    global_tr = (global_tr - global_mu) / global_sd

    norm = {
        "node_mu": node_mu, "node_sd": node_sd,
        "edge_mu": edge_mu, "edge_sd": edge_sd,
        "global_mu": global_mu, "global_sd": global_sd,
    }

    train_loader = DataLoader(
        GraphSeqDataset(node_tr, edge_tr, global_tr, y_dist_tr, time_tr),
        batch_size=batch_size,
        shuffle=True,
        drop_last=False
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = GraphGRUTimeDistPredictor(
        node_dim=node_tr.shape[-1],
        edge_dim=edge_tr.shape[-1],
        global_dim=global_tr.shape[-1],
        num_bins=y_dist_tr.shape[1],
        graph_hidden=128,
        msg_dim=128,
        graph_out_dim=128,
        gru_hidden=128,
        num_gru_layers=1,
        dropout=0.3,
    ).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    senders_t = torch.tensor(senders, dtype=torch.long, device=device)
    receivers_t = torch.tensor(receivers, dtype=torch.long, device=device)
    time_bin_centers_t = torch.tensor(time_bin_centers, dtype=torch.float32, device=device)

    for ep in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_mae = 0.0
        total_n = 0

        for node_x, edge_x, global_x, yb_dist, tb in train_loader:
            node_x = node_x.to(device)
            edge_x = edge_x.to(device)
            global_x = global_x.to(device)
            yb_dist = yb_dist.to(device)
            tb = tb.to(device)

            opt.zero_grad(set_to_none=True)
            logits = model(node_x, edge_x, global_x, senders_t, receivers_t)
            loss = soft_cross_entropy(logits, yb_dist)
            loss.backward()
            opt.step()

            with torch.no_grad():
                prob = torch.softmax(logits, dim=-1)
                pred_time = (prob * time_bin_centers_t.unsqueeze(0)).sum(dim=-1)
                batch_mae = torch.abs(pred_time - tb).mean().item()

            bs = node_x.size(0)
            total_loss += loss.item() * bs
            total_mae += batch_mae * bs
            total_n += bs

        print(f"epoch {ep:02d} | train_loss {total_loss / total_n:.6f} | train_mae {total_mae / total_n:.3f}")

    return model, norm, senders, receivers

@torch.no_grad()
def test_graph_model(
    model,
    data_test: dict,
    y_dist_test: np.ndarray,
    time_test: np.ndarray,
    norm,
    senders,
    receivers,
    time_bin_centers: np.ndarray,
    T: int = 100,
    batch_size: int = 512,
    use_masses: bool = False,
):
    node_te, edge_te, global_te, _, _ = build_graph_data(
        data_test, T=T, use_masses=use_masses
    )
    y_dist_te = np.asarray(y_dist_test).astype(np.float32)
    time_te = np.asarray(time_test).astype(np.float32)

    node_te = (node_te - norm["node_mu"]) / norm["node_sd"]
    edge_te = (edge_te - norm["edge_mu"]) / norm["edge_sd"]
    global_te = (global_te - norm["global_mu"]) / norm["global_sd"]

    device = next(model.parameters()).device
    loader = DataLoader(
        GraphSeqDataset(node_te, edge_te, global_te, y_dist_te, time_te),
        batch_size=batch_size,
        shuffle=False
    )

    return eval_graph_model(model, loader, device, senders, receivers, time_bin_centers)



In [ ]:
model, norm, senders, receivers = train_graph_model_on_trainset(
    data_train,
    y_soft_train,
    time_train,
    TIME_BIN_CENTERS,
    T=100,
    epochs=20,
    batch_size=256,
    lr=1e-3,
    use_masses=False,
)

metrics, pred_dist_test, pred_time_test, true_time_test = test_graph_model(
    model,
    data_test,
    y_soft_test,
    time_test,
    norm,
    senders,
    receivers,
    TIME_BIN_CENTERS,
    T=100,
    batch_size=512,
    use_masses=False,
)



In [ ]:
print(metrics)
print("pred_dist_test shape:", pred_dist_test.shape)
print("pred_time_test[:10]:", np.round(pred_time_test[:10], 2))
print("true_time_test[:10]:", np.round(true_time_test[:10], 2))

